# calibration-repro: train CIFAR-100 ResNets on Kaggle / Colab

Reproduces the training side of Guo et al. (2017), *On Calibration of Modern Neural Networks*.

**Kaggle:** Settings -> Accelerator *GPU T4 x2*, Internet *on*, then *Save Version -> Save & Run All* so the
run survives closing the browser. **Colab:** Runtime -> Change runtime type -> GPU.

A Kaggle session stops after 12 h and then keeps no output, so the 17 runs are split into four parts (`PART` in the
jobs cell); run one part per session and download `logits_part<PART>.zip` from the Output tab. Each run writes
`results/logits/<model>_wd<wd>_seed<s>.npz`. Estimated times (not measured): ResNet-110 ~3 h, ResNet-56 ~1.5 h per
run on one T4; part 1 also measures the real time (see `results/logs/`).

In [ ]:
import os, subprocess
REPO = "https://github.com/BenarmasNADIR/calibration-repro.git"
if not os.path.exists("calibration-repro"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("calibration-repro")
print(os.getcwd())

In [ ]:
# Training needs only torch, torchvision, numpy and pyyaml. Kaggle/Colab ship them (with a CUDA build of PyTorch that
# matches their drivers), so install from requirements.txt only what is missing; this avoids replacing the platform's
# numpy under its torch. The analysis (scipy, scikit-learn, matplotlib, pandas) runs locally with `uv sync`.
import importlib.util, re
need = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "yaml": "pyyaml"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
pins = [l.strip() for l in open("requirements.txt") if re.match(r"^(%s)==" % "|".join(missing), l)] if missing else []
print("missing:", missing or "none")
if pins:
    subprocess.run(["pip", "install", "-q", *pins], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Download CIFAR-100 once, before parallel jobs start (avoids two processes downloading at the same time).
!python -c "import sys; sys.path.insert(0, 'src'); from data import load_cifar100; load_cifar100('data')"  # ~170 MB

In [ ]:
# Optional sanity check (~1 min): CPU smoke test + unit tests.
!pip install -q pytest
!python src/train.py --config configs/smoke.yaml --device cpu
!python -m pytest -q

In [ ]:
# Choose what to run: one job = one training run (one line of arguments to src/train.py).
# One PART per Kaggle session (each well under the 12 h limit, even if the time estimates are 2x too low).
# PART = "all" runs all 17 jobs (~14 h on T4 x2, too long for one Kaggle session). Existing .npz files are skipped.
PART = "1"
WDS = ["1e-5", "1e-4", "5e-4", "1e-3", "5e-3"]
r110 = lambda seeds: [f"--config configs/resnet110_main.yaml --seed {s}" for s in seeds]
r56_wd = lambda s: [f"--config configs/wd_ablation.yaml --wd {wd} --seed {s}" for wd in WDS]
depth = lambda s: [f"--config configs/depth_ablation.yaml --depth {d} --seed {s}" for d in [20, 32]]
PARTS = {
    "1": r110([0, 1]),                     # main result, ResNet-110 seeds 0-1 (~3 h est.)
    "2": r110([2]) + depth(0) + depth(1),  # ResNet-110 seed 2 + ResNet-20/32 (~3 h est.)
    "3": r56_wd(0),                        # weight-decay ablation, ResNet-56, seed 0 (~4.5 h est.)
    "4": r56_wd(1),                        # weight-decay ablation, ResNet-56, seed 1 (~4.5 h est.)
}
PARTS["all"] = [j for k in "1234" for j in PARTS[k]]
# Depth ablation at depth 56/110 reuses the wd=1e-4 runs (same recipe and seeds).
JOBS = PARTS[PART]
print(f"part {PART}: {len(JOBS)} jobs")

In [ ]:
# Run the jobs, one per visible GPU in parallel (Kaggle T4 x2 -> 2 at a time). Logs in results/logs/.
import torch, time, pathlib
pathlib.Path("results/logs").mkdir(parents=True, exist_ok=True)
n_gpu = max(torch.cuda.device_count(), 1)
queue, running = list(JOBS), {}
while queue or running:
    for gpu in range(n_gpu):
        if gpu not in running and queue:
            job = queue.pop(0)
            log = open("results/logs/" + job.replace("--", "").replace("configs/", "").replace(" ", "_") + ".log", "a")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            running[gpu] = (subprocess.Popen(["python", "src/train.py", *job.split()],
                                             stdout=log, stderr=subprocess.STDOUT, env=env), job)
            print(f"GPU {gpu}: started {job}")
    for gpu, (p, job) in list(running.items()):
        if p.poll() is not None:
            print(f"GPU {gpu}: finished {job} (exit {p.returncode})")
            del running[gpu]
    time.sleep(30)
!tail -n 2 results/logs/*.log

In [ ]:
# Zip logits (+ logs) for download. Kaggle: Output tab. Colab: the download below.
ZIP = f"logits_part{PART}.zip"
!cd results && zip -qr ../../{ZIP} logits logs && ls -la ../../{ZIP}
try:
    from google.colab import files
    files.download(f"../{ZIP}")
except ImportError:
    pass

Then locally: unzip each `logits_part*.zip` into `results/` (so files land in `results/logits/`) and run
`uv run python src/analyze.py`.